# **1. Perkenalan Dataset**

Notebook ini mengikuti Template Eksperimen MSML. Dataset yang digunakan adalah **Titanic Dataset** untuk menyelesaikan masalah klasifikasi biner: memprediksi apakah penumpang selamat (`Survived = 1`) atau tidak selamat (`Survived = 0`).

Dataset berisi informasi penumpang seperti kelas tiket (`Pclass`), jenis kelamin (`Sex`), usia (`Age`), jumlah saudara/pasangan (`SibSp`), jumlah orang tua/anak (`Parch`), tarif (`Fare`), kabin (`Cabin`), dan pelabuhan keberangkatan (`Embarked`).

Tujuan eksperimen:
1. Memuat dataset Titanic.
2. Melakukan Exploratory Data Analysis (EDA).
3. Melakukan preprocessing manual.
4. Menyimpan dataset hasil preprocessing dalam bentuk `train.csv` dan `test.csv`.
5. Menjadikan notebook ini sebagai panduan untuk membuat file otomatisasi preprocessing.

# **2. Import Library**

Pada tahap ini, library yang digunakan mencakup `pandas`, `numpy`, `matplotlib`, `seaborn`, serta modul preprocessing dari `scikit-learn`.

In [ ]:
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# **3. Memuat Dataset**

Dataset dimuat dari folder `namadataset_raw`. Jika notebook dijalankan dari folder `preprocessing`, path yang digunakan adalah `../namadataset_raw/titanic.csv`.

In [ ]:
RAW_DATA_PATH = '../namadataset_raw/titanic.csv'
TARGET_COLUMN = 'Survived'

df = pd.read_csv(RAW_DATA_PATH)
df.head()

In [ ]:
print('Shape dataset:', df.shape)
df.info()

In [ ]:
df.describe(include='all')

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini dilakukan analisis awal untuk memahami karakteristik dataset Titanic, termasuk distribusi target, missing value, duplikasi data, distribusi fitur numerik, dan hubungan fitur dengan target.

In [ ]:
# Cek missing value
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# Cek data duplikat
print('Jumlah data duplikat:', df.duplicated().sum())

In [ ]:
# Distribusi target
print(df[TARGET_COLUMN].value_counts())
print(df[TARGET_COLUMN].value_counts(normalize=True))

sns.countplot(data=df, x=TARGET_COLUMN)
plt.title('Distribusi Target Survived')
plt.show()

In [ ]:
# Distribusi fitur kategorikal terhadap target
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x='Sex', hue=TARGET_COLUMN, ax=axes[0])
axes[0].set_title('Survived berdasarkan Sex')
sns.countplot(data=df, x='Pclass', hue=TARGET_COLUMN, ax=axes[1])
axes[1].set_title('Survived berdasarkan Pclass')
sns.countplot(data=df, x='Embarked', hue=TARGET_COLUMN, ax=axes[2])
axes[2].set_title('Survived berdasarkan Embarked')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f'Distribusi {col}')
    plt.show()

In [ ]:
# Korelasi fitur numerik
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='Blues')
plt.title('Correlation Heatmap')
plt.show()

# **5. Data Preprocessing**

Tahap preprocessing dilakukan untuk mengubah dataset mentah menjadi dataset yang siap dilatih oleh model machine learning. Tahapan yang dilakukan:

1. Menghapus data duplikat.
2. Membuat fitur baru `FamilySize` dan `IsAlone`.
3. Mengisi missing value pada `Age`, `Fare`, dan `Embarked`.
4. Menghapus kolom yang tidak digunakan: `PassengerId`, `Name`, `Ticket`, dan `Cabin`.
5. Membagi data menjadi train dan test.
6. Melakukan scaling fitur numerik dan encoding fitur kategorikal.
7. Menyimpan hasil preprocessing ke `train.csv`, `test.csv`, dan `preprocessing_metadata.json`.

In [ ]:
df_processed = df.copy()
df_processed = df_processed.drop_duplicates()

# Feature engineering
df_processed['FamilySize'] = df_processed['SibSp'] + df_processed['Parch'] + 1
df_processed['IsAlone'] = (df_processed['FamilySize'] == 1).astype(int)

# Handling missing value
df_processed['Age'] = df_processed['Age'].fillna(df_processed['Age'].median())
df_processed['Fare'] = df_processed['Fare'].fillna(df_processed['Fare'].median())
df_processed['Embarked'] = df_processed['Embarked'].fillna(df_processed['Embarked'].mode()[0])

# Drop kolom yang tidak digunakan
drop_columns = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_processed = df_processed.drop(columns=[col for col in drop_columns if col in df_processed.columns])

df_processed.head()

In [ ]:
# Split dataset
X = df_processed.drop(columns=[TARGET_COLUMN])
y = df_processed[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Embarked']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', make_one_hot_encoder(), categorical_features),
    ],
    remainder='drop'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = list(preprocessor.get_feature_names_out())

train_df = pd.DataFrame(X_train_processed, columns=feature_names)
train_df[TARGET_COLUMN] = y_train.reset_index(drop=True)

test_df = pd.DataFrame(X_test_processed, columns=feature_names)
test_df[TARGET_COLUMN] = y_test.reset_index(drop=True)

train_df.head()

In [ ]:
OUTPUT_DIR = 'namadataset_preprocessing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_path = os.path.join(OUTPUT_DIR, 'train.csv')
test_path = os.path.join(OUTPUT_DIR, 'test.csv')
metadata_path = os.path.join(OUTPUT_DIR, 'preprocessing_metadata.json')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

metadata = {
    'dataset_name': 'Titanic Dataset',
    'target_column': TARGET_COLUMN,
    'raw_shape': list(df.shape),
    'processed_train_shape': list(train_df.shape),
    'processed_test_shape': list(test_df.shape),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'drop_columns': drop_columns,
    'feature_names': feature_names,
    'created_at': datetime.now().isoformat(),
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)

print('Dataset preprocessing berhasil disimpan:')
print(train_path)
print(test_path)
print(metadata_path)

## Kesimpulan Preprocessing

Dataset Titanic telah diproses menjadi data numerik yang siap digunakan untuk training model. Seluruh proses pada notebook ini dikonversi ke dalam file `automate_MuhammadSidiqFirdaus.py` agar preprocessing dapat dijalankan secara otomatis melalui terminal maupun GitHub Actions.